# task_08 Solution

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

base = Path("../data")


In [ ]:

students = pd.read_csv(base / "students.csv")
courses = pd.read_csv(base / "courses.csv")
enrollments = pd.read_csv(base / "enrollments.csv")
attendance = pd.read_csv(base / "attendance.csv")
activities = pd.read_csv(base / "activities.csv")

enroll = enrollments.merge(courses, on="course_id")
weighted = enroll.assign(weighted_exam=enroll["exam_score"] * enroll["credit"]).groupby("student_id").agg(total_weighted=("weighted_exam", "sum"), total_credit=("credit", "sum"))
weighted["weighted_exam"] = weighted["total_weighted"] / weighted["total_credit"]

att = attendance.groupby("student_id").agg(absent=("absent_days", "sum"), school=("school_days", "sum"))
att["attendance_rate"] = (att["school"] - att["absent"]) / att["school"]

score = students.merge(weighted[["weighted_exam"]], left_on="student_id", right_index=True).merge(att[["attendance_rate"]], left_on="student_id", right_index=True)
score["penalized"] = score["weighted_exam"] * score["attendance_rate"]
q1 = str(score.groupby("homeroom")["penalized"].mean().sort_values(ascending=False).index[0])

stem_avg = enroll[enroll["subject"] == "STEM"].groupby("student_id")["exam_score"].mean()
club_counts = activities.groupby("student_id")["club"].nunique()
q2 = str(len(set(stem_avg[stem_avg >= 88].index) & set(club_counts[club_counts >= 2].index)))

algebra = enroll[enroll["course_name"] == "Algebra"][["student_id", "exam_score"]]
club_scores = activities.merge(algebra, on="student_id")
q3 = str(club_scores.groupby("club")["exam_score"].median().sort_values(ascending=False).index[0])

stud_diff = enrollments.groupby("student_id").agg(avg_homework=("homework_score", "mean"), avg_exam=("exam_score", "mean"))
stud_diff["diff"] = stud_diff["avg_homework"] - stud_diff["avg_exam"]
q4 = str(stud_diff.sort_values(["diff", "student_id"], ascending=[False, True]).index[0])

honor_students = set(weighted[weighted["weighted_exam"] >= 85].index)
q5 = str(attendance[attendance["student_id"].isin(honor_students)].groupby("month")["absent_days"].sum().sort_values(ascending=False).index[0])


Q1: If each student's weighted exam average uses course credits and is then multiplied by that student's overall attendance rate, which homeroom has the highest mean attendance-penalized score?

In [ ]:
print(q1)


Q2: How many students have a STEM-course exam average of at least 88 and belong to at least two clubs?

In [ ]:
print(q2)


Q3: Among club members, which club has the highest median Algebra exam score?

In [ ]:
print(q3)


Q4: Which student_id has the largest value of average_homework_score minus average_exam_score across enrolled courses?

In [ ]:
print(q4)


Q5: Define honor-roll students as those with weighted exam average >= 85. For honor-roll students, which month has the highest total absent_days?

In [ ]:
print(q5)
